# Домашнее задание 6

В этом задании мы:
1. Посмотрим на то, как получать эмбеддинги слов с помощью счетчиков и с помощью word2vec.
2. Построим простую сеть для классификации текста, замерим качество.
3. Посмотрим, как меняется качество этой сети в зависимости от качества эмбеддингов.
4. Построим TextCNN, обучим, сравним качество.

Работать будем с датасетом IMDb.

In [2]:
# Загрузка данных IMDb с помощью Keras
from keras.datasets import imdb

# Датасет хранится в двух кусках: сами данные и индексы (трейн/тест)
# Скачаем индексы
(X_train_indices, y_train), (X_test_indices, y_test) = imdb.load_data(num_words=10000)
# И сами данные. Это словарь вида {"слово": 0, "другое": 100}
# Выделим первые три слота под специальные токены: <PAD>, <START> и <UNK>
# Нумерация в исходном датасете идет с единицы, поэтому вычтем 1
word_to_index = {k: v - 1 + 3 for k, v in imdb.get_word_index().items()} | {
    "<PAD>": 0,
    "<START>": 1,
    "<UNK>": 2,
}
index_to_word = {index: word for word, index in word_to_index.items()}


# Функция для преобразования последовательности индексов в последовательность слов
def indices_to_words(indices):
    return " ".join(index_to_word.get(index, "<UNK>") for index in indices)


# Список строк, вида ["Good movie", "Bad film", ...]
X_train = [indices_to_words(indices) for indices in X_train_indices]
X_test = [indices_to_words(indices) for indices in X_test_indices]

В отзывах на фильмы скорее всего будут использовать различные сокращения фраз.
Например, AFAIK - "as far as I know".
Мы собрали словарь некоторых из них.

In [ ]:
chat_words = {
    "AFAIK": "As Far As I Know",
    "AFK": "Away From Keyboard",
    "ASAP": "As Soon As Possible",
    "ATK": "At The Keyboard",
    "ATM": "At The Moment",
    "A3": "Anytime, Anywhere, Anyplace",
    "BAK": "Back At Keyboard",
    "BBL": "Be Back Later",
    "BBS": "Be Back Soon",
    "BFN": "Bye For Now",
    "B4N": "Bye For Now",
    "BRB": "Be Right Back",
    "BRT": "Be Right There",
    "BTW": "By The Way",
    "B4": "Before",
    "CU": "See You",
    "CUL8R": "See You Later",
    "CYA": "See You",
    "FAQ": "Frequently Asked Questions",
    "FC": "Fingers Crossed",
    "FWIW": "For What It's Worth",
    "FYI": "For Your Information",
    "GAL": "Get A Life",
    "GG": "Good Game",
    "GN": "Good Night",
    "GMTA": "Great Minds Think Alike",
    "GR8": "Great!",
    "G9": "Genius",
    "IC": "I See",
    "ICQ": "I Seek you (also a chat program)",
    "ILU": "I Love You",
    "IMHO": "In My Honest/Humble Opinion",
    "IMO": "In My Opinion",
    "IOW": "In Other Words",
    "IRL": "In Real Life",
    "LDR": "Long Distance Relationship",
    "LTNS": "Long Time No See",
    "L8R": "Later",
    "MTE": "My Thoughts Exactly",
    "M8": "Mate",
    "NRN": "No Reply Necessary",
    "OIC": "Oh I See",
    "PITA": "Pain In The A..",
    "PRT": "Party",
    "PRW": "Parents Are Watching",
    "QPSA": "Que Pasa?",
    "ROFL": "Rolling On The Floor Laughing",
    "ROFLOL": "Rolling On The Floor Laughing Out Loud",
    "ROTFLMAO": "Rolling On The Floor Laughing My A.. Off",
    "SK8": "Skate",
    "STATS": "Your sex and age",
    "ASL": "Age, Sex, Location",
    "THX": "Thank You",
    "TTFN": "Ta-Ta For Now!",
    "TTYL": "Talk To You Later",
    "U2": "You Too",
    "U4E": "Yours For Ever",
    "WB": "Welcome Back",
    "WTF": "What The F...",
    "WTG": "Way To Go!",
    "WUF": "Where Are You From?",
    "W8": "Wait...",
    "7K": "Sick:-D Laughter",
    "TFW": "That feeling when",
    "MFW": "My face when",
    "MRW": "My reaction when",
    "IFYP": "I feel your pain",
    "LOL": "Laughing out loud",
    "TNTL": "Trying not to laugh",
    "JK": "Just kidding",
    "IDC": "I don’t care",
    "ILY": "I love you",
    "IMU": "I miss you",
    "ADIH": "Another day in hell",
    "ZZZ": "Sleeping, bored, tired",
    "WYWH": "Wish you were here",
    "BAE": "Before anyone else",
    "FIMH": "Forever in my heart",
    "BSAAW": "Big smile and a wink",
    "BWL": "Bursting with laughter",
    "LMAO": "Laughing my a** off",
    "BFF": "Best friends forever",
    "CSL": "Can’t stop laughing",
}
# Чтобы не иметь проблем с регистром, приведем все к нижнему регистру
chat_words = {key.lower(): value.lower() for key, value in chat_words.items()}

### Задание №1
Посчитайте, какое количество таких сокращений встречается в тестовом наборе данных `X_train`.

Например, для строк:
```
[
    "AFAIK, he is AFK",
    "Need to go, BBS",
    "Are you AFK ?",
]
```
ответ будет 4, т.к. всего в текстах встретилось 4 сокращения: AFAIK, AFK, BBS, AFK.

Сдайте в ЛМС одно число - количество сокращений. Например, `123`.

In [ ]:
type(X_train_indices)

In [ ]:
indices_to_words(X_train_indices[0])

In [ ]:
chat_words.keys()

In [ ]:
y_train.shape

In [ ]:
len(index_to_word.keys())

In [ ]:
from collections import Counter
import numpy as np

flat_words = [index_to_word.get(index) for index in np.concatenate(X_train_indices)]
print(len(flat_words))

# 1. Считаем частоту всех слов во втором списке
target_counter = Counter(flat_words)
# print(target_counter)

# 2. Создаем словарь с результатами
result = {}
for word in chat_words.keys():
    result[word] = target_counter.get(word, 0) # get вернет 0, если слова нет

print(result)
print(sum(result.values()))

In [ ]:
# решение авторов
import tqdm

cnt = 0
for text in tqdm.tqdm(X_train):
    for word in text.split():
        if word in chat_words:
            cnt += 1
cnt

Подумайте самостоятельно: стоит ли заменять эти сокращения в `X_train` и `X_test`?

## Простая модель, простой эмбеддинг
Обучим на этих данных классификатор.
Для этого надо слова превратить в вектора - _эмбеддинги_, затем построить модель.

### Задание №2

Обучите `CountVectorizer` из scikit-learn на текстах - это будет модель для эмбеддингов.
Используйте `max_features=10000`.

Затем обучите простую нейросеть (код ниже) на классификацию отзывов.

Сдайте в ЛМС код модели `SimpleNeuralNet` и файл с весами обученной модели (`model.pt`).

In [ ]:
X_train[-1]

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(max_features=2)
X = vectorizer.fit_transform(X_train)
print(vectorizer.get_feature_names_out())
print(X.toarray())

In [ ]:
print(type(X))

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import Adam
from torch.utils.data import DataLoader, TensorDataset, Dataset
from sklearn.feature_extraction.text import CountVectorizer
import tqdm


class SimpleNeuralNet(nn.Module):
    def __init__(self, input_dim: int):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_dim, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        out = self.fc(x)
        return out.squeeze()


vectorizer = CountVectorizer(max_features=10000)
X_train_bow = vectorizer.fit_transform(X_train)
X_test_bow = vectorizer.transform(X_test)

model = SimpleNeuralNet(input_dim=X_train_bow.shape[1])

In [ ]:
# Решение через Dataset, неоптимальное
class BowDataset(Dataset):
    def __init__(self, texts, labels):
        self.data = texts
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        label = torch.tensor(self.labels[index], dtype=torch.float32)
        word_matrix = torch.tensor(
            self.data[index].toarray(),
            dtype=torch.float32
        ).squeeze()

        return word_matrix, label

train_ds = BowDataset(X_train_bow, y_train)
test_ds = BowDataset(X_test_bow, y_test)

train_loader = DataLoader(
    train_ds,
    batch_size=32,
    shuffle=True,
    # num_workers=4
)

test_loader = DataLoader(
    test_ds,
    batch_size=32,
    shuffle=False,
    # num_workers=4
)

In [ ]:
# TensorDataset
X_train_dense = torch.tensor(X_train_bow.toarray(), dtype=torch.float32)
X_test_dense = torch.tensor(X_test_bow.toarray(), dtype=torch.float32)

y_train_t = torch.tensor(y_train, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32)

train_ds = TensorDataset(X_train_dense, y_train_t)
test_ds = TensorDataset(X_test_dense, y_test_t)

train_loader = DataLoader(
    train_ds,
    batch_size=512,
    shuffle=True
)

test_loader = DataLoader(
    test_ds,
    batch_size=512
)

In [ ]:
@torch.no_grad()
def loss_on_dataset(model: nn.Module, loader: DataLoader, device: torch.device):
    model.to(device).eval()
    loss = 0
    for x, label in loader:
        pred = model(x)
        loss = loss + criterion(pred, label)
    model.train()
    return (loss / len(loader)).item()


@torch.no_grad()
def accuracy(model: nn.Module, loader: DataLoader, device: torch.device):
    count_correct, count_total = 0, 0
    model.to(device).eval()
    for x, labels in loader:
        x = x.to(device)
        labels = labels.to(device).squeeze()
        pred_val = model(x).squeeze()
    
        pred_labels = (pred_val >= 0.5).float()
        count_correct += (pred_labels == labels).sum().item()
        count_total += len(labels)
        
    model.train()
    return count_correct / count_total

acc = []
criterion = nn.BCELoss()

def train_loop(model: nn.Module, 
               train_loader: DataLoader, 
               test_loader: DataLoader, 
               device: torch.device, 
               n_epochs: int):
    
    optimizer = Adam(model.parameters(), lr=1e-4)
    model.to(device)
    
    for epoch in range(n_epochs):
        print(f"Epoch {epoch}")
        model.train()
        for i, (X, label) in enumerate(tqdm.tqdm(train_loader)):
            optimizer.zero_grad()
            X = X.to(device)
            
            label = label.to(device).squeeze()
            pred = model(X)
            
            # print(pred.dtype, label.dtype)
            loss = criterion(pred, label)
            loss.backward()
            optimizer.step()
            if i % 10 == 0:
                accu = accuracy(model, test_loader, device)
                acc.append(accu)
                print(f"Итерация {i}: точность {accu}")
        

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu" 
# accuracy(model, test_loader, device)
# model = SimpleNeuralNet(input_dim=X_train_bow.shape[1])
train_loop(model, train_loader, test_loader, device, n_epochs=10)

In [ ]:
torch.save(model.state_dict(), "model.pt")

In [ ]:
# Решение авторов
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.feature_extraction.text import CountVectorizer  # isort: skip
from sklearn.metrics import classification_report


class SimpleNeuralNet(nn.Module):
    def __init__(self, input_dim: int):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_dim, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        out = self.fc(x)
        return out.squeeze()


vectorizer = CountVectorizer(max_features=10_000)
X_train_bow = vectorizer.fit_transform(X_train).toarray()
X_test_bow = vectorizer.transform(X_test).toarray()


def train_neural_net(X_train, y_train, X_test, y_test, input_dim):
    model = SimpleNeuralNet(input_dim)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    # Преобразуем данные в тензоры PyTorch
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
    y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

    train_data = TensorDataset(X_train_tensor, y_train_tensor)
    train_loader = DataLoader(train_data, batch_size=64, shuffle=True)

    # Обучение модели
    for epoch in range(50):
        model.train()
        for inputs, labels in train_loader:
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    # Тестирование модели
    model.eval()
    with torch.no_grad():
        outputs = model(X_test_tensor)
        preds = (outputs > 0.5).float()
        print("\nПростая нейронная сеть:")
        print(classification_report(y_test_tensor, preds))
    return model


model = train_neural_net(X_train_bow, y_train, X_test_bow, y_test, X_train_bow.shape[1])
# torch.save(model.state_dict(), "model.pt")

In [ ]:
# Почистим память перед дальнейшими экспериментами
del X_train_bow, X_test_bow, vectorizer

## Word2Vec

Попробуем использовать более продвинутую модель для подсчета эмбеддингов.
Возьмем `word2vec` - это популярная модель для эмбеддингов, ее часто используют в качестве бейзлайна.

In [5]:
from gensim.downloader import load

word2vec = load("word2vec-google-news-300")
word2vec.similar_by_word("cat")

[('cats', 0.8099379539489746),
 ('dog', 0.760945737361908),
 ('kitten', 0.7464985251426697),
 ('feline', 0.7326234579086304),
 ('beagle', 0.7150582671165466),
 ('puppy', 0.7075453400611877),
 ('pup', 0.6934291124343872),
 ('pet', 0.6891531348228455),
 ('felines', 0.6755931973457336),
 ('chihuahua', 0.6709762215614319)]

Про word2vec часто говорят, что он дает "осмысленные" эмбеддинги.
Давайте это проверим на примере.

### Задание №3
Какое слово получается, если сделать Paris - France + Germany, и с какой косинусной близостью?

Сдайте в ЛМС слово и близость в формате `'Munich', 0.98`.

In [ ]:
word2vec??

In [ ]:
paris_ind = word2vec.key_to_index["Paris"]
france_ind = word2vec.key_to_index["France"]
germany_ind = word2vec.key_to_index["Germany"]
vector = word2vec[paris_ind] - word2vec[france_ind] + word2vec[germany_ind]

word2vec.similar_by_vector(vector)

In [ ]:
# Решение авторов
word2vec.most_similar(word2vec["Paris"] - word2vec["France"] + word2vec["Germany"])

Можно надеяться, что если эмбеддинги будут более осмысленные, то качество модели увеличится.
Давайте проверять.

### Задание №4

Возьмите ту же модель `SimpleNeuralNet` и обучите её, взяв эмбеддинги из `word2vec`.

Вы, возможно, подумаете: `word2vec` выдает по вектору на каждое слово, а в предложении слов много.
Как с этим быть, предлагаем подумать самостоятельно.

Сдайте в ЛМС код `SimpleNeuralNet` и файл с весами модели `model.pt`.

In [6]:
import numpy as np
import tqdm


def sentence_to_embedding(sentence: str) -> np.ndarray:
    # На каждое предложение возвращаем 1 вектор размерности (300, )
    # Подумайте, что делать со словами, которых нет в словаре word2vec.
    vector = []
    for word in sentence.split():
        if word not in word2vec.key_to_index:
            continue
        vector.append(word2vec[word])
    return np.stack(vector).mean(0)



X_train_w2v = np.stack(
    [sentence_to_embedding(sentence) for sentence in tqdm.tqdm(X_train)]
)
X_test_w2v = np.stack(
    [sentence_to_embedding(sentence) for sentence in tqdm.tqdm(X_test)]
)
assert X_train_w2v.shape == (25_000, 300)
assert X_test_w2v.shape == (25_000, 300)

100%|██████████| 25000/25000 [00:10<00:00, 2435.10it/s]


### Me: TF-IDF
TF-IDF состоит из двух компонентов:

1. TF (Term Frequency) — Частота термина
Показывает, насколько часто слово встречается в данном документе.

Формула: TF(t, d) = (число вхождений слова t в документ d) / (общее число слов в документе d)

Смысл: Если слово "отличный" встречается в отзыве 3 раза из 50 слов, TF будет выше, чем если оно встретилось 1 раз.

2. IDF (Inverse Document Frequency) — Обратная частота документа
Показывает, насколько слово уникально или редко встречается во всем корпусе отзывов.

Формула: IDF(t, D) = log( (общее количество документов N) / (количество документов, содержащих слово t) )

Смысл: Если слово "отличный" встречается в 90% всех отзывов — IDF будет маленьким (log от числа близкого к 1). Если слово "безупречный" встречается только в 1% отзывов — IDF будет большим.

3. TF-IDF
Перемножаем два показателя:
TF-IDF = TF * IDF

In [ ]:
# Взвешенная сумма используя TF-IDF term frequency - inverse document frequency

from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
import gensim
from gensim.models import Word2Vec

# Допустим, у вас есть список отзывов (предложений)
corpus = X_train

# --- ШАГ 1: Считаем TF-IDF веса для слов ---
vectorizer = TfidfVectorizer()
# .fit_transform учится на корпусе и сразу превращает текст в матрицу TF-IDF
# Но нам нужны не числа, а веса слов, поэтому хитрее:
vectorizer.fit(corpus) 
# Получаем словарь: слово -> индекс
vocab = vectorizer.vocabulary_ 
# Получаем IDF веса для всех слов
idf_vals = vectorizer.idf_

# Создадим словарь вида {слово: tfidf_вес}
word_tfidf = {}
for word, idx in vocab.items():
    word_tfidf[word] = idf_vals[idx]

# --- ШАГ 2: Функция для получения взвешенного вектора предложения ---
def get_tfidf_weighted_sentence_vector(sentence, word2vec_dict, tfidf_dict):
    words = sentence.split()
    vectors = []
    weights = []
    
    for word in words:
        if word in word2vec_dict and word in tfidf_dict:
            vectors.append(word2vec_dict[word])
            weights.append(tfidf_dict[word]) # Используем вес IDF как важность
        # Если слова нет в word2vec или в tfidf (редко), пропускаем или добавляем нулевой вектор
    
    if not vectors: # Если ни одного слова не нашлось
        return np.zeros(next(iter(word2vec_dict.values())).shape)
    
    # Взвешенное суммирование
    weighted_sum = np.zeros_like(vectors[0])
    total_weight = sum(weights)
    for vec, weight in zip(vectors, weights):
        weighted_sum += vec * weight
    
    # Нормируем (делим на сумму весов)
    if total_weight > 0:
        return weighted_sum / total_weight
    else:
        return np.zeros_like(vectors[0])

# Проверим для первого предложения
sentence = corpus[0]
vector = get_tfidf_weighted_sentence_vector(sentence, word2vec, word_tfidf)
print(f"Вектор для '{sentence}': {vector}")

# В реальном коде вы бы прогнали все отзывы через эту функцию
# и получили бы матрицу признаков X для обучения классификатора.

In [ ]:
def sentence_to_embedding_tfidf(sentence: str) -> np.ndarray:
    # На каждое предложение возвращаем 1 вектор размерности (300, )
    # Подумайте, что делать со словами, которых нет в словаре word2vec.
    return get_tfidf_weighted_sentence_vector(sentence, word2vec, word_tfidf)

X_train_w2v_tfidf = np.stack(
    [sentence_to_embedding_tfidf(sentence) for sentence in tqdm.tqdm(X_train)]
)
X_test_w2v_tfidf = np.stack(
    [sentence_to_embedding_tfidf(sentence) for sentence in tqdm.tqdm(X_test)]
)
assert X_train_w2v_tfidf.shape == (25_000, 300)
assert X_test_w2v_tfidf.shape == (25_000, 300)

In [ ]:
# Повтор всего цикла
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.feature_extraction.text import CountVectorizer  # isort: skip
from sklearn.metrics import classification_report


class SimpleNeuralNet(nn.Module):
    def __init__(self, input_dim: int):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_dim, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        out = self.fc(x)
        return out.squeeze()


def train_neural_net(X_train, y_train, X_test, y_test, input_dim, param):
    model = SimpleNeuralNet(input_dim)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    # Преобразуем данные в тензоры PyTorch
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
    y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

    train_data = TensorDataset(X_train_tensor, y_train_tensor)
    train_loader = DataLoader(train_data, batch_size=64, shuffle=True)

    # Обучение модели
    for epoch in range(50):
        model.train()
        for inputs, labels in train_loader:
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    # Тестирование модели
    model.eval()
    with torch.no_grad():
        outputs = model(X_test_tensor)
        preds = (outputs > 0.5).float()
        print(f"\nПростая нейронная сеть c {param}:")
        print(classification_report(y_test_tensor, preds))
    return model

In [ ]:
model = train_neural_net(X_train_w2v, y_train, X_test_w2v, y_test, X_train_w2v.shape[1], param="word2vec")
# torch.save(model.state_dict(), "model.pt")

In [ ]:
model_tfidf = train_neural_net(X_train_w2v_tfidf, y_train, X_test_w2v_tfidf, y_test, X_train_w2v_tfidf.shape[1], param="word2vec tfidf")
# torch.save(model.state_dict(), "model.pt")

Вывод: tfidf не улучшил качество по сравнению с обычнм word2vec mean. Качество по сравнению с BOW также снизилось. Видимо, дело в простоте модели и размерности embedding (BOW ~ 10000 против W2V == 300)

In [ ]:
...
torch.save(model.state_dict(), "model-w2v.pt")

In [ ]:
# Решение авторов
import numpy as np


def sentence_to_embedding(sentence: str) -> np.ndarray:
    # На каждое предложение возвращаем 1 вектор размерности (300, )
    # Подумайте, что делать со словами, которых нет в словаре word2vec.
    rv = []
    for word in sentence.split():
        if word not in word2vec.key_to_index:
            continue
        rv.append(word2vec[word])
    return np.stack(rv).mean(0)


X_train_w2v = np.stack(
    [sentence_to_embedding(sentence) for sentence in tqdm.tqdm(X_train)]
)
X_test_w2v = np.stack(
    [sentence_to_embedding(sentence) for sentence in tqdm.tqdm(X_test)]
)
assert X_train_w2v.shape == (25_000, 300)
assert X_test_w2v.shape == (25_000, 300)

In [ ]:
del X_train_w2v, X_test_w2v

Возможно, качество вырастет не так сильно, но зато теперь мы можем считать эмбеддинги отдельно для каждого слова.
А это значит, что мы можем попробовать учесть порядок слов в предложении.

## TextCNN

Модель TextCNN - одна из первых попыток учесть положение слов в тексте.
Попробуем обучить эту модель.

### Задание №5

Обучите `TextCNN` на датасете IMDb, используя эмбеддинги `word2vec` и паддинг из прошлого пункта.

Не забудьте, что в таком подходе каждое предложение имеет размерность не `(300, )`, а `(n_words, 300)`.
А это значит:
- нужен паддинг;
- нужно переписать логику подсчета эмбеддингов для предложения.

Вы можете поискать подсказки в семинаре.

Сдайте в ЛМС код `TextCNN` и файл с весами модели.

In [70]:
import torch
import torch.nn as nn
from collections.abc import Collection
from gensim.models import KeyedVectors
import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import classification_report
from torch.utils.data import Dataset

filter_sizes = [2, 3, 4]
num_filters = 2


class TextCnn(nn.Module):
    def __init__(self):
        super().__init__()
        emb_size = 300
        self.text_filters = nn.ModuleList(
            [
                nn.Sequential(
                    nn.Conv1d(emb_size, num_filters, kernel_size=filter_sizes[i]),
                    nn.ReLU(),
                )
                for i in range(len(filter_sizes))
            ]
        )
        
        self.fc = nn.Linear(
            num_filters * len(filter_sizes),
            1,
        )

    def forward(self, x: torch.Tensor):
        to_fc = []
        rv = []
        for f in self.text_filters:
            # Свертка - выдаст [(bs, 2, n - 1), (bs, 2, n - 2), (bs, 2, n - 3)]
            result = f(x.permute((0, 2, 1)))
            # MaxPooling вдоль предложения - даст [(bs, 2), (bs, 2), (bs, 2)]
            result = torch.max(result, dim=2)[0]
            rv.append(result)
            
        x = torch.concat(rv, dim=1)
        to_fc.append(x)

        x = torch.hstack(to_fc)
        x = self.fc(x).squeeze(1)
        return x

def pad_or_trim_to_length(
    pad_length: int, vectors: list[np.ndarray], pad_vector: np.ndarray
):
    assert pad_vector.ndim == 1
    # допаддим слева, а если превысили - обрежем с конца
    vectors = vectors[:pad_length] + [pad_vector] * max(0, pad_length - len(vectors))
    return np.stack(vectors)


def seq_to_emb(
    sentences: Collection[str], wv: KeyedVectors, pad_length: int, pad_token: str = "##"
):
    rv = []
    for sentence in tqdm.tqdm(sentences):
        sentence_embeddings = []
        for one_token in sentence.split():
            if wv.has_index_for(one_token):
                sentence_embeddings.append(wv.get_vector(one_token))
        sentence_embeddings = pad_or_trim_to_length(
            pad_length, sentence_embeddings, wv.get_vector(pad_token)
        )
        rv.append(np.stack(sentence_embeddings))
#     return np.stack(rv)


def train_neural_net(train_loader, test_loader, device):
    model = TextCnn().to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    # Обучение модели
    for epoch in tqdm.trange(1):
        model.train()
        for inputs, labels in ейвьtrain_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    # Тестирование модели
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs = model(inputs)
            preds = (torch.sigmoid(outputs) > 0.5).float()
    
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    print(classification_report(all_labels, all_preds))
    return model


# class ImdbDataset(Dataset):
#     def __init__(self, texts, labels, wv, pad_length=1000):
#         self.texts = texts
#         self.labels = labels
#         self.wv = wv
#         self.pad_length = pad_length

#     def __len__(self):
#         return len(self.texts)

#     def __getitem__(self, idx):
#         sentence = self.texts[idx]

#         vectors = []
#         for token in sentence.split():
#             if self.wv.has_index_for(token):
#                 vectors.append(self.wv.get_vector(token))

#         vectors = pad_or_trim_to_length(
#             self.pad_length,
#             vectors,
#             self.wv.get_vector("##")
#         )

#         return (
#             torch.tensor(vectors, dtype=torch.float32),
#             torch.tensor(self.labels[idx], dtype=torch.float32)
#         )

class ImdbDataset(Dataset):
    def __init__(self, texts, labels, wv, pad_length=1000):
        self.texts = texts
        self.labels = labels
        self.pad_length = pad_length

        # быстрые структуры
        self.key_to_index = wv.key_to_index
        self.vectors = wv.vectors
        self.pad_vector = wv["##"]

        self.emb_dim = self.vectors.shape[1]

    def __len__(self):
        return len(self.texts)

    def sent_emb_fast(self, sentence):
        tokens = sentence.split()
        # индексы слов
        idxs = [
            self.key_to_index[t] for t in tokens if t in self.key_to_index
        ]

        idxs = idxs[:self.pad_length]

        if len(idxs) > 0:
            emb = self.vectors[idxs]
        else:
            emb = np.empty((0, self.emb_dim))

        if len(idxs) < self.pad_length:
            pad = np.tile(
                self.pad_vector,
                (self.pad_length - len(idxs), 1)
            )

            emb = np.vstack([emb, pad])
        return emb

    def __getitem__(self, idx):
        emb = self.sent_emb_fast(self.texts[idx])
        return (
            torch.from_numpy(emb).float(),
            torch.tensor(self.labels[idx], dtype=torch.float32),
        )

In [ ]:
n = None
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_dataset = ImdbDataset(X_train[:n], y_train[:n], word2vec, pad_length=1000)
test_dataset = ImdbDataset(X_test[:n], y_test[:n], word2vec, pad_length=1000)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=64, num_workers=4)

train_neural_net(train_loader, test_loader, device)

In [72]:
torch.save(model.state_dict(), "model_s.pt")

In [65]:
# оптимизация с подготовкой индексов векторов
import numpy as np
import torch
from torch.utils.data import Dataset
import torch.nn as nn
from sklearn.metrics import classification_report
from torch.utils.data import DataLoader

wv = word2vec
vectors = wv.vectors
key_to_index = wv.key_to_index
vocab_size, emb_dim = vectors.shape
embedding_matrix = torch.tensor(vectors)

class ImdbDataset(Dataset):
    def __init__(self, texts, labels, key_to_index, pad_length=400):
        self.texts = texts
        self.labels = labels
        self.key_to_index = key_to_index
        self.pad_length = pad_length

    def __len__(self):
        return len(self.texts)
    
    def encode(self, sentence):
        tokens = sentence.split()
        ids = [
            self.key_to_index[t]
            for t in tokens
            if t in self.key_to_index
        ]
        ids = ids[:self.pad_length]
        if len(ids) < self.pad_length:
            ids += [0] * (self.pad_length - len(ids))
        return ids

    def __getitem__(self, idx):
        ids = self.encode(self.texts[idx])
        return (
            torch.tensor(ids, dtype=torch.long),
            torch.tensor(self.labels[idx], dtype=torch.float32),
        )


class TextCNN(nn.Module):
    def __init__(self, embedding_matrix):
        super().__init__()
        vocab_size, emb_dim = embedding_matrix.shape
        self.embedding = nn.Embedding.from_pretrained(
            embedding_matrix,
            freeze=True
        )

        filter_sizes = [2,3,4]
        num_filters = 2

        self.convs = nn.ModuleList([
            nn.Conv1d(
                emb_dim,
                num_filters,
                kernel_size=k
            )
            for k in filter_sizes
        ])
        
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(
            num_filters * len(filter_sizes),
            1
        )

    def forward(self, x):
        x = self.embedding(x)
        x = x.permute(0,2,1)
        conv_out = []
        
        for conv in self.convs:
            c = self.relu(conv(x))
            p = torch.max(c, dim=2)[0]
            conv_out.append(p)

        x = torch.cat(conv_out, dim=1)
        # x = self.dropout(x)
        x = self.fc(x)
        return x.squeeze(1)


train_dataset = ImdbDataset(
    X_train,
    y_train,
    key_to_index,
    pad_length=400
)

test_dataset = ImdbDataset(
    X_test,
    y_test,
    key_to_index,
    pad_length=400
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64)

In [67]:
# Запуск оптимизированной модели 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TextCNN(embedding_matrix).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in tqdm.trange(1):
    model.train()
    for x, y in train_loader:
        x = x.to(device)
        y = y.to(device)
        outputs = model(x)
        loss = criterion(outputs, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for x, y in tqdm.tqdm(test_loader):
        x = x.to(device)
        y = y.to(device)
        outputs = model(x)
        preds = (torch.sigmoid(outputs) > 0.5).float()
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.cpu().numpy())

print(classification_report(all_labels, all_preds))

100%|██████████| 391/391 [00:03<00:00, 104.62it/s]

              precision    recall  f1-score   support

         0.0       0.71      0.72      0.72     12500
         1.0       0.72      0.71      0.71     12500

    accuracy                           0.71     25000
   macro avg       0.71      0.71      0.71     25000
weighted avg       0.71      0.71      0.71     25000



In [68]:
torch.save(model.state_dict(), "Text_CNN_model3.pt")

In [3]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 53.3 MB/s eta 0:00:00:00:0100:01


In [4]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report
import tqdm

from gensim.downloader import load

word2vec = load("word2vec-google-news-300")

wv = word2vec
vectors = wv.vectors
key_to_index = wv.key_to_index
vocab_size, emb_dim = vectors.shape
embedding_matrix = torch.tensor(vectors)

# -----------------------------
# 3️⃣ Dataset
# -----------------------------
class ImdbDataset(Dataset):
    def __init__(self, texts, labels, key_to_index, pad_length=400):
        self.texts = texts
        self.labels = labels
        self.key_to_index = key_to_index
        self.pad_length = pad_length

    def __len__(self):
        return len(self.texts)

    def encode(self, sentence):
        tokens = sentence.split()
        ids = [
            self.key_to_index[t] for t in tokens if t in self.key_to_index
        ]
        ids = ids[:self.pad_length]
        if len(ids) < self.pad_length:
            ids += [0] * (self.pad_length - len(ids))
        return ids

    def __getitem__(self, idx):
        ids = self.encode(self.texts[idx])
        return (
            torch.tensor(ids, dtype=torch.long),
            torch.tensor(self.labels[idx], dtype=torch.float32),
        )

train_dataset = ImdbDataset(X_train, y_train, key_to_index, pad_length=400)
test_dataset = ImdbDataset(X_test, y_test, key_to_index, pad_length=400)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64)

# -----------------------------
# 4️⃣ Модель TextCNN без хранения Word2Vec
# -----------------------------
class TextCNN(nn.Module):
    def __init__(self, embedding_matrix, device):
        super().__init__()
        self.embedding_matrix = embedding_matrix.to(device)  # переносим на устройство
        self.device = device

        emb_dim = embedding_matrix.shape[1]
        filter_sizes = [2,3,4]
        num_filters = 100

        self.convs = nn.ModuleList([
            nn.Conv1d(emb_dim, num_filters, kernel_size=k)
            for k in filter_sizes
        ])
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(num_filters * len(filter_sizes), 1)

    def forward(self, x):
        # x: [batch, seq_len]
        x = self.embedding_matrix[x]          # [batch, seq_len, emb_dim]
        x = x.permute(0,2,1)                  # [batch, emb_dim, seq_len]
        conv_out = []
        for conv in self.convs:
            c = self.relu(conv(x))
            p = torch.max(c, dim=2)[0]
            conv_out.append(p)
        x = torch.cat(conv_out, dim=1)
        x = self.dropout(x)
        x = self.fc(x)
        return x.squeeze(1)

# -----------------------------
# 5️⃣ Обучение
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TextCNN(embedding_matrix, device).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

def train(model):
    for epoch in tqdm.trange(10, desc="Training"):
        model.train()
        for x, y in train_loader:
            x = x.to(device)
            y = y.to(device)
            optimizer.zero_grad()
            outputs = model(x)
            loss = criterion(outputs, y)
            loss.backward()
            optimizer.step()

    # -----------------------------
    # 6️⃣ Оценка
    # -----------------------------
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for x, y in tqdm.tqdm(test_loader, desc="Evaluating"):
            x = x.to(device)
            y = y.to(device)
            outputs = model(x)
            preds = (torch.sigmoid(outputs) > 0.5).float()
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    print(classification_report(all_labels, all_preds))

: 

: 

: 

In [ ]:
train(model)

NameError: name 'train' is not defined

In [74]:
torch.save(model.state_dict(), "Text_CNN_model_compact.pt")

In [7]:
word2vec.index_to_key[:10]

['</s>', 'in', 'for', 'that', 'is', 'on', '##', 'The', 'with', 'said']

In [10]:
word_count = 0
pad = "##"
for sent in X_train + X_test:
    if pad in sent:
        word_count += 1
print(word_count)

0


In [30]:
import numpy as np
sent_count = np.array([len(x) for x in X_train + X_test])
print(f"max: {sent_count.max()}, mean: {sent_count.mean():.1f}")
print(f"Reviews with >= 1000 words: {sum(sent_count >= 1000)}")

max: 14122, mean: 1231.0
Reviews with >= 1000 words: 22374


### Задание №6
Поэкспериментируйте с моделью и данными, попробуйте выбить наибольшее качество.

Сдайте в ЛМС наибольший accuracy, который у вас получился.